# 🎯 [QLT04_M06] 부대별/장비별 가동률 추세 및 가동률 저하 요인 분석 대시보드 구현
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m06_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M06 |
| **미션 주제** | 부대별/장비별 가동률 추세 및 가동률 저하 요인 분석 대시보드 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 06. 부대별/장비별 가동률 추세 및 가동률 저하 요인 분석 대시보드 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

부대별/장비별 가동률 추세 및 저하 요인 분석 대시보드를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **부대별·장비 유형별 가동률 추세와 저하 요인을 보여 주는 대시보드를 만들고, 비정상 입력에도 중단되지 않도록 입력 검증기와 에러 핸들러를 갖춘다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **12개월 가상 추세로 부대별 월 기울기, 최근 3개월 평균, 저하 경고와 부대 x 유형 가동률, 대기 손실일 구성을 표와 그래프로 보여 준다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/utils/qlt04_validator.py 및 에러 핸들러` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 사용자에게 알기 쉬운 에러 메시지 및 조치 가이드 제공

#### 미션 PASS 조건

- 정량 기준: 예외 상황 발생 시 크래시(Crash) 0건, 에러 로그 기록률 100%


- 진행 기준: 예외 주입 테스트 10종 통과 및 감사 로그 파일 생성


- 검증 방법: 비정상 데이터(결측, 특수문자, 대용량) 스트레스 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""부대별·장비 유형별 12개월 가동률 추세와 저하 요인 구성을 텍스트 대시보드와 그래프로 보여 주고, 결측·특수문자·자료형 오류·대용량·빈 자료·열 누락 등 예외 13종을 주입해도 중단 없이 오류 격리·안내 메시지·오류 로그·재시도·롤백이 동작하는지 시험하는 입력 검증기와 에러 핸들러를 구현한다."""
import copy
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 장비 가동률·정비 대기 자료 생성기 (실제 부대·장비·제원·가동률 정보 없음)
# 장비 유형 A(궤도형)·B(화포형)·C(차륜형)은 추상 구분이며 실제 장비와 무관하다.
# 최근 90일 동안 장비 1대의 정비 대기 요인(부품·정비병·외주 정비·외주 검사·재작업)과 가동률을 만든다.
# ---------------------------------------------------------------------------
UNITS = ["가상A부대", "가상B부대", "가상C부대", "가상D부대", "가상E부대", "가상F부대"]
ETYPES = ["유형A", "유형B", "유형C"]
WINDOW = 90                                       # 관측 기간(일)
CAUSES = ["부품 대기", "정비병 부족", "외주 정비", "검사·재작업"]
CAUSE_COL = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "검사·재작업": "inspect_wait_days"}
NUM_FIELDS = ["age_years", "op_hours", "maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count",
              "spec_mismatch", "defect_reports", "parts_stock_pct", "mechanic_fill_pct", "avail_pct"]
FEAT = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports", "etype_B", "etype_C"]
FEAT_KO = {"age_years": "장비 연령", "maint_orders": "정비 요청 건수", "parts_stock_pct": "부품 재고 가용률", "mechanic_fill_pct": "정비병 충원율", "outsource_days": "외주 정비 일수",
           "inspect_wait_days": "외주 검사 대기 일수", "rework_count": "재작업 건수", "spec_mismatch": "규격·제원 불일치", "defect_reports": "하자·불만 접수",
           "etype_B": "유형B 여부", "etype_C": "유형C 여부"}


def unit_profile(seed=3):
    """부대별 고정 특성: 부품 재고 수준, 정비병 충원율, 외주 정비소 거리(일수 가중), 외주 검사 혼잡도."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({"unit": UNITS, "stock": r.uniform(45, 95, len(UNITS)), "fill": r.uniform(55, 98, len(UNITS)),
                         "dist": r.uniform(0.8, 1.6, len(UNITS)), "queue": r.uniform(0.7, 2.2, len(UNITS))})


def make_fleet(n=300, seed=7, shock=0.0):
    """장비 n대의 90일 관측 자료(필드 16개)와 정답 요인을 만든다. shock은 외주 검사 병목 악화 정도(AAR용 환경 변화)."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    ui = rng.integers(0, len(UNITS), n)
    et = rng.choice(ETYPES, n, p=[.35, .3, .35])
    age = np.clip(rng.gamma(4, 2.2, n), 0.5, 30).round(1)
    eff = np.array([{"유형A": 1.15, "유형B": 1.0, "유형C": 0.9}[t] for t in et])
    orders = rng.poisson(np.clip(1.2 + 0.12 * age, 0.5, 6) * eff)
    stock = np.clip(pf["stock"].to_numpy()[ui] + rng.normal(0, 8, n), 20, 100).round(0)
    fill = np.clip(pf["fill"].to_numpy()[ui] + rng.normal(0, 6, n), 30, 100).round(0)
    parts_w = (orders * rng.gamma(2, 0.55 * (1 + ((100 - stock) / 28) ** 2) / 2)).round(0)
    labor_w = (orders * rng.gamma(2, 0.7 * (1 + ((100 - fill) / 24) ** 2) / 2)).round(0)
    n_out = rng.binomial(orders, 0.3)
    out_d = (n_out * rng.gamma(3, 4.5 * pf["dist"].to_numpy()[ui] / 3)).round(0)
    insp_w = (n_out * rng.gamma(2, (2.6 + shock) * pf["queue"].to_numpy()[ui] / 2)).round(0)
    spec = rng.poisson(0.2 + 0.04 * age)
    rework = rng.binomial(n_out, np.clip(0.08 + 0.1 * spec, 0, .6))
    defect = rng.poisson(0.3 + 0.7 * rework)
    base_rep = orders * 1.8
    down = base_rep + parts_w + labor_w + out_d + insp_w + rework * 4 + rng.normal(0, 1.2, n)
    comp = np.column_stack([parts_w, labor_w, out_d, insp_w + rework * 4])
    df = pd.DataFrame({"equip_id": [f"E-{i + 1:04d}" for i in range(n)], "unit": np.array(UNITS)[ui], "etype": et, "age_years": age,
                       "op_hours": np.clip(rng.normal(320 - 4 * age, 70, n), 40, 600).round(0), "maint_orders": orders,
                       "parts_wait_days": parts_w, "labor_wait_days": labor_w, "outsource_days": out_d, "inspect_wait_days": insp_w,
                       "rework_count": rework, "spec_mismatch": spec, "defect_reports": defect, "parts_stock_pct": stock, "mechanic_fill_pct": fill,
                       "avail_pct": (100 * (1 - np.clip(down, 0, WINDOW) / WINDOW)).round(1)})
    df["main_cause"] = np.where(comp.max(axis=1) < 3, "없음", np.array(CAUSES)[comp.argmax(axis=1)])
    return df


def feat_frame(df):
    """회귀·분석용 설명 변수 표(유형은 더미 변수)."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    return x[FEAT]


def make_trend(months=12, seed=5, shock_unit=None):
    """[교육용가상자료] 부대 x 유형 x 월 평균 가동률 추세. shock_unit 부대는 하반기에 외주 검사 병목으로 가동률이 떨어진다."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    rows = []
    for _, u in pf.iterrows():
        base = 62 + 0.22 * u["stock"] + 0.12 * u["fill"] - 3.0 * u["queue"]
        for t in ETYPES:
            off = {"유형A": -3.0, "유형B": 0.0, "유형C": 2.0}[t]
            for m in range(months):
                v = base + off + 2.0 * np.sin(m / 12 * 2 * np.pi) + rng.normal(0, 1.0)
                if shock_unit == u["unit"] and m >= months // 2:
                    v -= 1.5 * (m - months // 2 + 1)
                rows.append((u["unit"], t, m + 1, round(float(np.clip(v, 30, 99)), 1)))
    return pd.DataFrame(rows, columns=["unit", "etype", "month", "avail_pct"])

fleet = make_fleet(300, seed=41)
trend = make_trend(12, seed=5, shock_unit="가상C부대")

# 1) 대시보드: 부대별 12개월 가동률 추세, 최근 3개월 평균, 월 기울기, 경고
SP = "▁▂▃▄▅▆▇█"
def spark(v):
    lo, hi = min(v), max(v)
    return "".join(SP[int((x - lo) / (hi - lo + 1e-9) * 7)] for x in v)
rows = []
for u, g in trend.groupby("unit"):
    m = g.groupby("month")["avail_pct"].mean()
    slope = np.polyfit(m.index, m.values, 1)[0]
    last3 = m.iloc[-3:].mean()
    rows.append({"부대": u, "추세(1~12월)": spark(m.values), "상반기": m.iloc[:6].mean(), "하반기": m.iloc[6:].mean(), "최근3개월": last3, "월 기울기(%p)": slope,
                 "경고": "저하 경고" if slope < -0.8 or last3 < 70 else ""})
dash = pd.DataFrame(rows)
print("[대시보드 1: 부대별 가동률 추세(가상)]")
print(dash.round(1).to_string(index=False))
t_et = trend[trend["month"] > 9].groupby(["unit", "etype"])["avail_pct"].mean().unstack().round(1)
print("\n[대시보드 2: 최근 3개월 부대 x 장비 유형 가동률]"); print(t_et.to_string())
lossc = ["parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days"]
loss = fleet.groupby("unit")[lossc].mean().round(1)
loss.columns = ["부품 대기", "정비병 부족", "외주 정비", "외주 검사 대기"]
print("\n[대시보드 3: 부대별 평균 대기 손실일(저하 요인 구성)]")
print(pd.concat([loss, loss.idxmax(axis=1).rename("최대 요인")], axis=1).to_string())
worst = fleet.nsmallest(3, "avail_pct")[["equip_id", "unit", "etype", "avail_pct", "main_cause"]]
print("[가동률 최하위 장비 3대]"); print(worst.to_string(index=False))

# 2) 입력 검증기 + 에러 핸들러
LOG = []
GUIDE = {"E101": "필수 열이 없습니다. 열 이름을 양식과 비교해 추가하세요.", "E102": "빈 자료입니다. 파일이 올바른지 확인하고 다시 올리세요.", "E201": "형식·범위 오류 행을 격리했습니다. 격리 목록을 확인해 원본을 수정하세요.",
         "E202": "중복 장비 번호를 1건만 남겼습니다. 최신 기록이 맞는지 확인하세요.", "E301": "일시적 읽기 오류입니다. 자동 재시도 후에도 실패하면 잠시 뒤 다시 시도하세요.",
         "E401": "처리 중 오류가 있어 변경을 취소(롤백)했습니다. 이전 화면 상태가 유지됩니다.", "E501": "자료 형식을 알 수 없습니다. 표 형식(열 이름 포함) 자료를 올리세요."}


def log(code, msg, level="ERROR"):
    LOG.append({"seq": len(LOG) + 1, "level": level, "code": code, "msg": msg, "guide": GUIDE[code]})


class UserError(Exception):
    def __init__(self, code, msg):
        super().__init__(msg); self.code = code


def validate_frame(df):
    """행 단위 검증: 형식(장비 번호), 허용값(부대), 범위·자료형(숫자), 중복. 불량 행은 격리하고 나머지는 계속 처리한다."""
    if not isinstance(df, pd.DataFrame):
        raise UserError("E501", f"DataFrame이 아닌 입력({type(df).__name__})")
    if len(df) == 0:
        raise UserError("E102", "행 0건")
    miss = [c for c in ["equip_id", "unit", "etype", "avail_pct", "maint_orders"] if c not in df.columns]
    if miss:
        raise UserError("E101", f"누락 열 {miss}")
    d = df.copy()
    num = pd.to_numeric(d["avail_pct"], errors="coerce"), pd.to_numeric(d["maint_orders"], errors="coerce")
    bad = (~d["equip_id"].astype(str).str.match(r"^E-\d{4,6}$")) | (~d["unit"].isin(UNITS)) | (~d["etype"].isin(ETYPES))
    bad |= ~np.isfinite(num[0]) | (num[0] < 0) | (num[0] > 100) | num[1].isna() | (num[1] < 0)
    q = d[bad]
    if len(q):
        log("E201", f"형식·범위 오류 {len(q)}행 격리(예: {str(q.iloc[0]['equip_id'])[:12]!r})", "WARN")
    d = d[~bad].copy()
    d["avail_pct"] = pd.to_numeric(d["avail_pct"]); d["maint_orders"] = pd.to_numeric(d["maint_orders"])
    dup = d.duplicated("equip_id")
    if dup.any():
        log("E202", f"중복 장비 번호 {int(dup.sum())}건 정리", "WARN")
    return d[~dup], q


class Store:
    """화면 상태 저장소: 갱신은 사본에서 먼저 하고 모두 성공해야 반영(트랜잭션 롤백)한다."""
    def __init__(self):
        self.state = {"version": 0, "avg": None}

    def commit(self, fn):
        snap = copy.deepcopy(self.state)
        try:
            fn(self.state); self.state["version"] += 1
            return True
        except Exception as e:
            self.state = snap; log("E401", f"갱신 실패 {type(e).__name__}: {str(e)[:40]}"); return False


STORE = Store()
FLAKY = {"n": 0}


def flaky_loader(df):
    FLAKY["n"] += 1
    if FLAKY["n"] <= 2:
        raise IOError("temporary read error")
    return df


def run(df, loader=None, fail_update=False):
    """대시보드 갱신 1회. 어떤 입력에서도 예외를 밖으로 던지지 않고 (상태, 처리 행수)를 돌려준다."""
    try:
        for attempt in range(3):                                         # 재시도(최대 3회)
            try:
                src = loader(df) if loader else df; break
            except IOError:
                log("E301", f"읽기 오류 재시도 {attempt + 1}/3", "WARN")
        else:
            raise UserError("E301", "재시도 3회 실패")
        clean, q = validate_frame(src)
        def upd(s):
            s["avg"] = float(clean["avail_pct"].mean())
            if fail_update:
                raise RuntimeError("render failure")
        ok = STORE.commit(upd)
        return ("처리" if not len(q) else "격리 후 처리") if ok else "롤백", len(clean)
    except UserError as e:
        log(e.code, str(e)); return "거부(안내)", 0
    except Exception as e:                                               # 예상 못 한 오류도 중단하지 않고 기록
        log("E501", f"{type(e).__name__}: {str(e)[:40]}"); return "거부(안내)", 0


good = fleet[["equip_id", "unit", "etype", "avail_pct", "maint_orders"]]
def mod(**kw):
    d = good.copy()
    for k, v in kw.items():
        d[k] = d[k].astype(object); d.loc[d.index[:3], k] = v
    return d
big = pd.DataFrame({"equip_id": [f"E-{i:06d}" for i in range(200000)], "unit": "가상A부대", "etype": "유형A",
                    "avail_pct": np.random.default_rng(1).uniform(40, 100, 200000), "maint_orders": 2})
big.loc[big.index[::1000], "avail_pct"] = np.nan
cases = [("가동률 결측(NaN)", mod(avail_pct=np.nan)), ("가동률 문자열 '많음'", mod(avail_pct="많음")), ("가동률 음수·100 초과", mod(avail_pct=-5)),
         ("장비 번호 특수문자", mod(equip_id="E-00;DROP --")), ("가상에 없는 부대명", mod(unit="미상부대")), ("무한대(inf) 값", mod(avail_pct=np.inf)),
         ("중복 장비 번호", pd.concat([good, good.head(5)])), ("필수 열 누락", good.drop(columns=["unit"])), ("빈 자료", good.iloc[0:0]),
         ("표가 아닌 입력(None)", None), ("대용량 20만 행(결측 포함)", big), ("읽기 오류 2회 후 성공(재시도)", good), ("갱신 중 오류(롤백)", good)]
print("\n[예외 주입 시험]")
crash = passed = 0
t0 = time.perf_counter()
for name, df in cases:
    n0 = len(LOG)
    try:
        st, n = run(df, loader=flaky_loader if "재시도" in name else None, fail_update="롤백" in name)
    except Exception as e:
        crash += 1; st, n = f"CRASH {e}", 0
    logged = len(LOG) > n0
    passed += (not st.startswith("CRASH")) and logged
    print(f"  {name:22s}-> {st:8s} {n:6d}행 남음 | 로그 {'기록' if logged else '없음'}" + (f" | {LOG[-1]['code']}: {LOG[-1]['guide'][:28]}" if logged else ""))
dt = time.perf_counter() - t0
print(f"예외 {len(cases)}종: 크래시 {crash}건(목표 0건: {'달성' if crash == 0 else '미달'}), 오류 로그 기록 {passed}/{len(cases)} ({passed / len(cases):.0%}, 목표 100%), 전체 {dt:.1f}초")
print(f"오류 로그 {len(LOG)}건 중 사용자 안내문 포함 {sum(bool(x['guide']) for x in LOG)}건, 상태 저장소 버전 {STORE.state['version']}(롤백 시 이전 값 유지), 평균 가동률 {STORE.state['avg']:.1f}%")
print("로그 예시:", LOG[0]["code"], LOG[0]["msg"], "->", LOG[0]["guide"])
print("※ 격리된 행의 수정과 경고 판단은 담당자가 확인한다. 로그는 메모리에 두며 실제 도구에서는 감사 로그 파일로 남긴다.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for u, g in trend.groupby("unit"):
    m = g.groupby("month")["avail_pct"].mean(); ax[0].plot(m.index, m.values, label='unit ' + u[2])
ax[0].set_title("Monthly availability by unit (synthetic)"); ax[0].set_xlabel("month"); ax[0].legend(ncol=3, fontsize=7)
loss.plot(kind="bar", stacked=True, ax=ax[1], legend=False); ax[1].set_xticklabels([u[2] for u in loss.index], rotation=0); ax[1].set_title("Mean wait-loss days by cause")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[대시보드 1: 부대별 가동률 추세(가상)]
   부대    추세(1~12월)  상반기  하반기  최근3개월  월 기울기(%p)    경고
가상A부대 ▃▇▆▇▇▅▃▂▁▂▁▁ 77.3 74.8   74.6       -0.3      
가상B부대 ▃▅▆▅▇▄▂▂▁▁▂▁ 79.4 76.9   76.8       -0.3      
가상C부대 ▆▇▇▇▇▇▆▅▃▂▁▁ 89.4 81.9   79.1       -1.2 저하 경고
가상D부대 ▆▅▆▇▆▄▃▁▁▂▁▃ 80.2 77.6   77.7       -0.3      
가상E부대 ▃▅▇▆▆▇▃▁▁▁▂▂ 79.2 76.1   76.1       -0.3      
가상F부대 ▄▆▇▇▅▆▄▄▂▁▂▃ 83.7 81.0   80.6       -0.3      

[대시보드 2: 최근 3개월 부대 x 장비 유형 가동률]
etype   유형A   유형B   유형C
unit                   
가상A부대  72.3  75.1  76.5
가상B부대  73.3  78.2  78.8
가상C부대  76.3  78.8  82.2
가상D부대  74.8  77.8  80.4
가상E부대  73.6  76.1  78.7
가상F부대  78.5  79.9  83.3

[대시보드 3: 부대별 평균 대기 손실일(저하 요인 구성)]
       부품 대기  정비병 부족  외주 정비  외주 검사 대기   최대 요인
unit                                         
가상A부대    5.0     3.5    3.3       2.2   부품 대기
가상B부대    4.3     5.7    3.2       2.1  정비병 부족
가상C부대    2.1     2.0    4.0       1.3   외주 정비
가상D부대    2.5     5.1    5.3       3.5   외주 정비
가상E부대    5.5     4.2    2.6       1.2   부품 대기
가상F부대    2.7     2.4    4.9       1.7   외주 정비
[가동률 최하위 장비 3대]
equip_id  unit etype  avail_pct main_cause
  E-0113 가상D부대   유형A       17.5      외주 정비
  E-0165 가상D부대   유형A       26.3     검사·재작업
  E-0283 가상A부대   유형A       30.2      외주 정비

[예외 주입 시험]
  가동률 결측(NaN)           -> 격리 후 처리     297행 남음 | 로그 기록 | E201: 형식·범위 오류 행을 격리했습니다. 격리 목록을 확
  가동률 문자열 '많음'          -> 격리 후 처리     297행 남음 | 로그 기록 | E201: 형식·범위 오류 행을 격리했습니다. 격리 목록을 확
  가동률 음수·100 초과         -> 격리 후 처리     297행 남음 | 로그 기록 | E201: 형식·범위 오류 행을 격리했습니다. 격리 목록을 확
  장비 번호 특수문자            -> 격리 후 처리     297행 남음 | 로그 기록 | E201: 형식·범위 오류 행을 격리했습니다. 격리 목록을 확
  가상에 없는 부대명            -> 격리 후 처리     297행 남음 | 로그 기록 | E201: 형식·범위 오류 행을 격리했습니다. 격리 목록을 확
... (이하 생략)
```
</details>
